# BODAQS IMU Inertial Explorer - Self-scoped

Explore persisted fused IMU inertial streams from one BODAQS library. This notebook is intentionally limited to time-series and frequency-distribution inspection. `roll_rad`, `pitch_rad`, and `yaw_enu_rad` are the persisted fixed-interval world-attitude solution; the optional short-gap stitch remains a separate display-only experiment.

Reprocess a BDQ with the preprocess-profile `imu_attitude.enabled` setting before using this notebook.

## 1. Configure library

Set `LIBRARIES_ROOT` and `LIBRARY_ID`, then run the cells top-to-bottom.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import ipywidgets as W
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'bodaqs_analysis').is_dir():
            return candidate
        analysis = candidate / 'analysis'
        if (analysis / 'bodaqs_analysis').is_dir():
            return analysis
    raise RuntimeError('Could not find the BODAQS analysis package root from the current working directory.')


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / 'OneDrive' / 'BODAQS-data'
LIBRARY_ID = 'archie'

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item['library_id']: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ', '.join(sorted(libraries)) or 'none found'
    raise ValueError(f'Library {LIBRARY_ID!r} was not found. Available libraries: {available}')

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]['root'])
pio.renderers.default = 'notebook_connected'
print(f'Analysis package root: {ANALYSIS_DIR}')
print(f'Library root: {LIBRARY_ROOT}')

Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\libraries\archie


## 2. Select processed sessions

Only physical sessions are shown. Select one or more sessions, then use **Load selected attitude streams** below. Sessions without a persisted attitude stream are reported but ignored.

In [2]:
from bodaqs_analysis.widgets.session_selector import make_session_selector

sel = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(sel['ui'])

## 3. Load an attitude stream

The stream selector identifies the session and IMU sensor. The inspector uses the persisted QC report in session metadata, so its status remains visible after reload.

In [3]:
from bodaqs_analysis.artifacts import load_session_artifacts
from bodaqs_analysis.attitude import INERTIAL_STREAM_SCHEMA, LEGACY_ATTITUDE_STREAM_SCHEMA

ATTITUDE_STREAMS = {}
stream_dd = W.Dropdown(description='Attitude stream', layout=W.Layout(width='720px'))
load_button = W.Button(description='Load selected attitude streams', button_style='primary')
load_status = W.Output(layout=W.Layout(width='100%'))


def _is_inertial_stream(name, metadata):
    return (
        str(name).startswith(('inertial_', 'attitude_'))
        or isinstance(metadata, dict) and metadata.get('schema') in {INERTIAL_STREAM_SCHEMA, LEGACY_ATTITUDE_STREAM_SCHEMA}
    )


def load_selected_attitude_streams(_=None):
    global ATTITUDE_STREAMS
    loaded, unavailable = {}, []
    for ref in sel['get_selected']():
        artifacts = load_session_artifacts(sel['store'], run_id=ref['run_id'], session_id=ref['session_id'])
        stream_dfs = artifacts.get('stream_dfs', {})
        stream_meta = artifacts.get('secondary_stream_meta', {})
        found = False
        for name, frame in stream_dfs.items():
            metadata = stream_meta.get(name, {})
            if _is_inertial_stream(name, metadata):
                key = f"{ref['session_id']} — {name}"
                loaded[key] = {
                    'df': frame.copy(),
                    'metadata': metadata,
                    'session_meta': artifacts.get('meta', {}),
                    'run_id': ref['run_id'],
                    'session_id': ref['session_id'],
                    'stream_name': name,
                    'imu_df': stream_dfs.get(metadata.get('imu_stream_name')),
                }
                found = True
        if not found:
            unavailable.append(ref['session_id'])

    ATTITUDE_STREAMS = loaded
    stream_dd.options = [(label, label) for label in loaded]
    stream_dd.value = next(iter(loaded), None)
    if loaded and 'refresh_attitude_inspector' in globals():
        refresh_attitude_inspector()
    with load_status:
        load_status.clear_output()
        print(f'Loaded {len(loaded)} attitude stream(s).')
        if unavailable:
            print('No persisted attitude stream: ' + ', '.join(unavailable))
            print('Reprocess these sessions with imu_attitude.enabled = true.')


load_button.on_click(load_selected_attitude_streams)
display(W.VBox([W.HBox([load_button, stream_dd]), load_status]))

## 4. Inspect time series and distributions

`*_rad` attitude and innovation signals are displayed in degrees. The time-series renderer limits only its displayed samples; distribution statistics use every finite sample in the selected time window. `Continuous yaw` and short-gap stitching operate only on the in-memory display copy.

In [4]:
signal_select = W.SelectMultiple(description='Signals', rows=12, layout=W.Layout(width='430px'))
start_time = W.FloatText(description='Start [s]', layout=W.Layout(width='190px'))
end_time = W.FloatText(description='End [s]', layout=W.Layout(width='190px'))
max_samples = W.BoundedIntText(value=12000, min=500, max=200000, step=500, description='Plot samples')
histogram_bins = W.BoundedIntText(value=80, min=10, max=500, step=10, description='Histogram bins')
yaw_display = W.ToggleButtons(
    options=[('Wrapped yaw', 'wrapped'), ('Continuous yaw', 'unwrapped')],
    value='wrapped',
    description='Yaw display',
)
stitch_short_gaps = W.Checkbox(value=False, description='Experimentally stitch short gaps', indent=False)
max_stitch_gap_s = W.BoundedFloatText(value=0.25, min=0.01, max=1.0, step=0.01, description='Max gap [s]')
refresh_button = W.Button(description='Refresh inspector')
plot_button = W.Button(description='Render selected views', button_style='primary')
metadata_out = W.Output(layout=W.Layout(width='100%'))
time_series_out = W.Output(layout=W.Layout(width='100%'))
distribution_out = W.Output(layout=W.Layout(width='100%'))


def _numeric_signals(frame):
    excluded = {'time_s', 'continuity_segment'}
    return [name for name in frame.columns if name not in excluded and pd.api.types.is_numeric_dtype(frame[name])]


def _display_signal(name, values):
    values = pd.to_numeric(values, errors='coerce')
    if name.endswith('_rad'):
        return name[:-4] + ' [deg]', np.degrees(values)
    return name, values


def _normalise_quaternion(quaternion):
    norm = float(np.linalg.norm(quaternion))
    return quaternion / norm if norm and np.isfinite(norm) else np.array([1.0, 0.0, 0.0, 0.0])


def _quaternion_multiply(left, right):
    lw, lx, ly, lz = left
    rw, rx, ry, rz = right
    return np.array([
        lw * rw - lx * rx - ly * ry - lz * rz,
        lw * rx + lx * rw + ly * rz - lz * ry,
        lw * ry - lx * rz + ly * rw + lz * rx,
        lw * rz + lx * ry - ly * rx + lz * rw,
    ])


def _quaternion_inverse(quaternion):
    norm_sq = float(np.dot(quaternion, quaternion))
    return np.array([quaternion[0], -quaternion[1], -quaternion[2], -quaternion[3]]) / norm_sq


def _quaternion_from_rotation_vector(rotation_vector):
    angle = float(np.linalg.norm(rotation_vector))
    if angle < 1.0e-12:
        return np.array([1.0, 0.0, 0.0, 0.0])
    axis = rotation_vector / angle
    return np.r_[np.cos(angle / 2.0), axis * np.sin(angle / 2.0)]


def _euler_zyx(quaternion):
    w, x, y, z = _normalise_quaternion(quaternion)
    rotation = np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ])
    return np.array([
        np.arctan2(rotation[2, 1], rotation[2, 2]),
        np.arcsin(np.clip(-rotation[2, 0], -1.0, 1.0)),
        np.arctan2(rotation[1, 0], rotation[0, 0]),
    ])


def _nearest_gyro(imu, time_s):
    times = pd.to_numeric(imu['time_s'], errors='coerce').to_numpy(float)
    index = int(np.searchsorted(times, time_s, side='left'))
    candidates = [candidate for candidate in (index - 1, index) if 0 <= candidate < len(imu)]
    nearest = min(candidates, key=lambda candidate: abs(times[candidate] - time_s))
    return imu.iloc[nearest][['body_gyro_x_rad_s', 'body_gyro_y_rad_s', 'body_gyro_z_rad_s']].to_numpy(float)


def _display_attitude_frame(item):
    frame = item['df'].copy()
    frame['display_continuity_segment'] = frame['continuity_segment']
    frame['display_gap_stitched_before'] = False
    stitched = []
    quaternion_columns = [f'q_body_to_world_enu_{axis}' for axis in 'wxyz']
    imu = item.get('imu_df')
    can_stitch = (
        stitch_short_gaps.value
        and isinstance(imu, pd.DataFrame)
        and all(column in imu.columns for column in ['time_s', 'body_gyro_x_rad_s', 'body_gyro_y_rad_s', 'body_gyro_z_rad_s'])
        and all(column in frame.columns for column in quaternion_columns)
    )
    groups = [group.index.to_numpy() for _, group in frame.groupby('continuity_segment', sort=False)]
    display_segment = 0
    for position, indices in enumerate(groups):
        if position == 0:
            frame.loc[indices, 'display_continuity_segment'] = display_segment
            continue
        previous = groups[position - 1]
        gap_s = float(frame.loc[indices[0], 'time_s'] - frame.loc[previous[-1], 'time_s'])
        bridged = False
        if can_stitch and 0.0 < gap_s <= max_stitch_gap_s.value:
            previous_q = frame.loc[previous[-1], quaternion_columns].to_numpy(float)
            original_first_q = frame.loc[indices[0], quaternion_columns].to_numpy(float)
            gyro = (_nearest_gyro(imu, float(frame.loc[previous[-1], 'time_s'])) + _nearest_gyro(imu, float(frame.loc[indices[0], 'time_s']))) / 2.0
            predicted_q = _normalise_quaternion(_quaternion_multiply(previous_q, _quaternion_from_rotation_vector(gyro * gap_s)))
            adjustment = _quaternion_multiply(predicted_q, _quaternion_inverse(original_first_q))
            quaternions = frame.loc[indices, quaternion_columns].to_numpy(float)
            corrected = np.vstack([_normalise_quaternion(_quaternion_multiply(adjustment, quaternion)) for quaternion in quaternions])
            frame.loc[indices, quaternion_columns] = corrected
            euler = np.vstack([_euler_zyx(quaternion) for quaternion in corrected])
            frame.loc[indices, ['roll_rad', 'pitch_rad', 'yaw_enu_rad']] = euler
            frame.loc[indices, 'display_gap_stitched_before'] = True
            stitched.append({'gap_s': gap_s, 'source_after_segment': int(frame.loc[indices[0], 'continuity_segment'])})
            bridged = True
        if not bridged:
            display_segment += 1
        frame.loc[indices, 'display_continuity_segment'] = display_segment

    if yaw_display.value == 'unwrapped' and 'yaw_enu_rad' in frame:
        for _, group in frame.groupby('display_continuity_segment', sort=False):
            frame.loc[group.index, 'yaw_enu_rad'] = np.unwrap(group['yaw_enu_rad'].to_numpy(float))
    frame.attrs['display_stitches'] = stitched
    return frame


def _window(frame):
    if frame.empty or 'time_s' not in frame:
        return frame.iloc[0:0].copy()
    time = pd.to_numeric(frame['time_s'], errors='coerce')
    return frame.loc[time.between(start_time.value, end_time.value, inclusive='both')].copy()


def _display_sample(frame, limit):
    if len(frame) <= limit:
        return frame
    segment_column = 'display_continuity_segment' if 'display_continuity_segment' in frame else 'continuity_segment'
    if segment_column not in frame:
        return frame.iloc[np.linspace(0, len(frame) - 1, limit, dtype=int)]
    groups = list(frame.groupby(segment_column, sort=False))
    minimum = max(2, limit // max(1, len(groups)))
    parts = []
    for _, group in groups:
        count = min(len(group), max(minimum, round(limit * len(group) / len(frame))))
        parts.append(group.iloc[np.linspace(0, len(group) - 1, count, dtype=int)])
    return pd.concat(parts, ignore_index=True)


def refresh_attitude_inspector(_=None):
    if not stream_dd.value or stream_dd.value not in ATTITUDE_STREAMS:
        return
    item = ATTITUDE_STREAMS[stream_dd.value]
    frame = item['df']
    time = pd.to_numeric(frame['time_s'], errors='coerce')
    signals = _numeric_signals(frame)
    defaults = [name for name in ('roll_rad', 'pitch_rad', 'yaw_enu_rad', 'linear_accel_body_x_m_s2', 'linear_accel_body_y_m_s2', 'course_innovation_rad') if name in signals]
    signal_select.options = signals
    signal_select.value = tuple(defaults or signals[:min(4, len(signals))])
    start_time.value = float(time.min())
    end_time.value = float(time.max())
    attitude_qc = item['session_meta'].get('attitude_qc', {})
    sensor = item['metadata'].get('sensor', item['stream_name'].removeprefix('inertial_').removeprefix('attitude_'))
    qc = attitude_qc.get(sensor, {}) if isinstance(attitude_qc, dict) else {}
    with metadata_out:
        metadata_out.clear_output()
        print(f"Run: {item['run_id']} | Session: {item['session_id']} | Stream: {item['stream_name']}")
        print(f"Samples: {len(frame):,} | Time: {time.min():.3f} to {time.max():.3f} s")
        print(f"Attitude status: {qc.get('status', 'not recorded')} | Yaw observed fraction: {qc.get('yaw_observed_fraction', float('nan')):.3f}")
        print(f"Gravity updates accepted: {qc.get('gravity_updates_accepted', 'not recorded')} | Course updates accepted: {qc.get('course_updates_accepted', 'not recorded')}")
        imu_name = item['metadata'].get('imu_stream_name')
        imu_metadata = item['session_meta'].get('secondary_streams', {}).get(imu_name, {})
        print(f"Gyro bias correction: {imu_metadata.get('gyro_bias_correction') or 'not applied'}")


def render_selected_views(_=None):
    if not stream_dd.value or stream_dd.value not in ATTITUDE_STREAMS:
        raise RuntimeError('Load and select an attitude stream first.')
    selected_signals = list(signal_select.value)
    if not selected_signals:
        raise ValueError('Select at least one signal.')
    display_frame = _display_attitude_frame(ATTITUDE_STREAMS[stream_dd.value])
    frame = _window(display_frame)
    if frame.empty:
        raise ValueError('The selected time range contains no attitude samples.')
    plotted = _display_sample(frame, max_samples.value)

    with time_series_out:
        time_series_out.clear_output()
        figure = make_subplots(rows=len(selected_signals), cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=selected_signals)
        group_column = 'display_continuity_segment' if 'display_continuity_segment' in plotted else None
        for row, signal in enumerate(selected_signals, start=1):
            label, values = _display_signal(signal, plotted[signal])
            groups = plotted.groupby(group_column, sort=False) if group_column else [(None, plotted)]
            for _, group in groups:
                _, group_values = _display_signal(signal, group[signal])
                figure.add_trace(go.Scattergl(x=group['time_s'], y=group_values, mode='lines', name=label, legendgroup=signal, showlegend=(row == 1)), row=row, col=1)
            figure.update_yaxes(title_text=label, row=row, col=1)
        figure.update_xaxes(title_text='Time [s]', row=len(selected_signals), col=1)
        stitch_note = f"; display-stitched gaps: {len(display_frame.attrs.get('display_stitches', []))}" if stitch_short_gaps.value else ''
        figure.update_layout(height=max(360, 260 * len(selected_signals)), title=f'Time series — {stream_dd.value}{stitch_note}', hovermode='x unified')
        figure.show()

    with distribution_out:
        distribution_out.clear_output()
        figure = make_subplots(rows=1, cols=2, subplot_titles=('Frequency distribution', 'Empirical cumulative distribution'))
        for signal in selected_signals:
            label, values = _display_signal(signal, frame[signal])
            values = np.asarray(values, dtype=float)
            values = values[np.isfinite(values)]
            if not len(values):
                continue
            figure.add_trace(go.Histogram(x=values, nbinsx=histogram_bins.value, histnorm='probability density', name=label, opacity=0.55), row=1, col=1)
            sorted_values = np.sort(values)
            figure.add_trace(go.Scattergl(x=sorted_values, y=np.arange(1, len(sorted_values) + 1) / len(sorted_values), mode='lines', name=label, legendgroup=signal, showlegend=False), row=1, col=2)
        figure.update_xaxes(title_text='Signal value', row=1, col=1)
        figure.update_xaxes(title_text='Signal value', row=1, col=2)
        figure.update_yaxes(title_text='Density', row=1, col=1)
        figure.update_yaxes(title_text='Cumulative fraction', range=[0, 1], row=1, col=2)
        figure.update_layout(height=480, barmode='overlay', title=f'Distributions — {start_time.value:.3f} to {end_time.value:.3f} s')
        figure.show()


stream_dd.observe(refresh_attitude_inspector, names='value')
refresh_button.on_click(refresh_attitude_inspector)
plot_button.on_click(render_selected_views)
controls = W.VBox([
    W.HBox([signal_select, W.VBox([start_time, end_time, max_samples, histogram_bins, yaw_display, stitch_short_gaps, max_stitch_gap_s, refresh_button, plot_button])]),
    metadata_out,
    time_series_out,
    distribution_out,
])
display(controls)

## Reading the first-slice attitude signals

- `roll_rad`, `pitch_rad`, and `yaw_enu_rad` are the body-to-world estimate. Yaw is world-frame only while the state is `world_enu_constrained` or `world_enu_degraded`.
- `yaw_sigma_deg` shows the propagated yaw uncertainty.
- `gravity_update_weight`, `course_update_weight`, innovations, and rejection codes show which corrections were usable.
- `attitude_state_code`: `0` gravity aligned; `1` world ENU constrained; `2` world ENU degraded after a prior course correction.

For definitions and acceptance rules, see `docs/analysis/BMI270_Attitude_Derived_Product.md`.